In [1]:
import lucene

In [2]:
lucene.VERSION

'10.0.0'

In [3]:
!ls

LICENSE  bin  extensions  node	node_modules  out  package.json  product.json


In [4]:
!pwd

/root/.vscode-server/bin/e3a5acfb517a443235981655413d566533107e92


In [5]:
!ls /mnt

wiki_movie_plots_deduped.csv


In [6]:
import pandas

ModuleNotFoundError: No module named 'pandas'

In [8]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 3.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 4.6 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]


In [9]:
import pandas

In [10]:
df = pandas.read_csv("/mnt/wiki_movie_plots_deduped.csv", sep=',')

In [ ]:
docid = 1
for _, row in df.iterrows():
    print("{} - {}".format(docid, row['Title']))
    docid += 1

1 - Kansas Saloon Smashers
2 - Love by the Light of the Moon
3 - The Martyred Presidents
4 - Terrible Teddy, the Grizzly King
5 - Jack and the Beanstalk
6 - Alice in Wonderland
7 - The Great Train Robbery
8 - The Suburbanite
9 - The Little Train Robbery
10 - The Night Before Christmas
11 - Dream of a Rarebit Fiend
12 - From Leadville to Aspen: A Hold-Up in the Rockies
13 - Kathleen Mavourneen
14 - Daniel Boone
15 - How Brown Saw the Baseball Game
16 - Laughing Gas
17 - The Adventures of Dollie
18 - The Black Viper
19 - A Calamitous Elopement
20 - The Call of the Wild
21 - A Christmas Carol
22 - The Fight for Freedom
23 - At the Altar
24 - A Drunkard's Reformation
25 - The Golden Louis
26 - The Lure of the Gown
27 - An Arcadian Maid
28 - A Christmas Carol
29 - Frankenstein
30 - Hemlock Hoax, the Detective
31 - The House with Closed Shutters
32 - A Lad from Old Ireland
33 - Pocahontas
34 - Ramona
35 - What the Daisy Said
36 - The Wonderful Wizard of Oz
37 - Baseball and Bloomers
38 - The

In [14]:
lucene.initVM()

Sep 30, 2025 9:38:26 PM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


In [15]:
!ls /mnt

wiki_movie_plots_deduped.csv


In [16]:
!ls /mnt

foo.txt  wiki_movie_plots_deduped.csv


# Indexing the Kaggle Movie Plot Dataset using PyLucene

In [18]:
from org.apache.lucene.store import FSDirectory
from java.nio.file import Paths

indexPath = "/mnt/movie_index"
indexDir = FSDirectory.open(Paths.get(indexPath))

In [21]:
from org.apache.lucene.analysis.en import EnglishAnalyzer
from org.apache.lucene.index import IndexWriter, IndexWriterConfig

analyzer = EnglishAnalyzer()
config = IndexWriterConfig(analyzer)
writer = IndexWriter(indexDir, config)

In [42]:
from org.apache.lucene.document import Document, IntField, TextField, StringField, StoredField, Field, FieldType
from org.apache.lucene.index import IndexOptions

indexType = FieldType()
indexType.setStored(True)
indexType.setTokenized(True)
indexType.setStoreTermVectors(True)
indexType.setIndexOptions(IndexOptions.DOCS_AND_FREQS_AND_POSITIONS)
indexType.setStoreTermVectorPositions(True)
indexType.setStoreTermVectorOffsets(False)

def indexDocument(movie):

    doc = Document() # one Lucene document

    # 1. Year
    year = int(movie['Release Year'])
    doc.add(IntField("YEAR", year, Field.Store.YES))

    # 2. Title
    title = movie['Title']
    doc.add(TextField("TITLE", title, Field.Store.YES))

    # 3. Origin
    origin = movie['Origin/Ethnicity']
    doc.add(StringField("ORIGIN", origin, Field.Store.YES))

    # 4. Director
    director = movie['Director']
    doc.add(TextField("DIRECTOR", director, Field.Store.YES))

    # 5. Cast
    cast = str(movie['Cast'])
    doc.add(TextField("CAST", cast, Field.Store.YES))

    # 6. Genre
    genre = movie['Genre']
    doc.add(StringField("GENRE", genre, Field.Store.YES))

    # 7. Wiki page
    wiki = movie['Wiki Page']
    doc.add(StoredField("URL", wiki))
    
    # 8. Plot
    plot = movie['Plot']
    doc.add(Field("PLOT", plot, indexType))
    
    return doc

In [43]:
docid = 1
for _, row in df.iterrows():
    print("{} - {}".format(docid, row['Title']))
    docid += 1
    doc = indexDocument(row)
    writer.addDocument(doc)

1 - Kansas Saloon Smashers
2 - Love by the Light of the Moon
3 - The Martyred Presidents
4 - Terrible Teddy, the Grizzly King
5 - Jack and the Beanstalk
6 - Alice in Wonderland
7 - The Great Train Robbery
8 - The Suburbanite
9 - The Little Train Robbery
10 - The Night Before Christmas
11 - Dream of a Rarebit Fiend
12 - From Leadville to Aspen: A Hold-Up in the Rockies
13 - Kathleen Mavourneen
14 - Daniel Boone
15 - How Brown Saw the Baseball Game
16 - Laughing Gas
17 - The Adventures of Dollie
18 - The Black Viper
19 - A Calamitous Elopement
20 - The Call of the Wild
21 - A Christmas Carol
22 - The Fight for Freedom
23 - At the Altar
24 - A Drunkard's Reformation
25 - The Golden Louis
26 - The Lure of the Gown
27 - An Arcadian Maid
28 - A Christmas Carol
29 - Frankenstein
30 - Hemlock Hoax, the Detective
31 - The House with Closed Shutters
32 - A Lad from Old Ireland
33 - Pocahontas
34 - Ramona
35 - What the Daisy Said
36 - The Wonderful Wizard of Oz
37 - Baseball and Bloomers
38 - The

In [44]:
writer.close()

In [45]:
!ls /mnt

foo.txt  movie_index  wiki_movie_plots_deduped.csv


In [46]:
!ls /mnt/movie_index

_0.cfe	_0.si	_1.cfs	_2.cfe	_2.si	    write.lock
_0.cfs	_1.cfe	_1.si	_2.cfs	segments_1
